# Final assignment: The BC Forest Finder
## Old Forest near Prince George

Begin by entering parameters below. Use the [VRI Data Dictionary](https://www2.gov.bc.ca/gov/content/industry/forestry/managing-our-forest-resources/forest-inventory/data-management-and-access/vri-data-standards#vridictionary) to help find desired attributes, or the table from the [BC Data Catalogue](https://catalogue.data.gov.bc.ca/dataset/vri-2024-forest-vegetation-composite-rank-1-layer-r1-/resource/0b45b37d-59e5-415f-8dff-99024c9d9264).

- The [BEC Zones and Subzones Guide](https://www.for.gov.bc.ca/hre/becweb/resources/classificationreports/subzones/index.html) may also be useful.
- For formatting the DuckDB WHERE clause, the following page may be useful:
    - [WHERE Clause (DuckDB)](https://duckdb.org/docs/stable/sql/query_syntax/where)

# Inputs
## Core Inputs
These three inputs, described below, are the most important part of our analysis. 

In [1]:
# Enter a string for a location to be geocoded, such as an address or name of
#   a town.
starting_location_string = "Prince George, BC"

# Enter the radius around the point to search for forest polygons (in 
#   kilometers). Note that this will be buffered by 30% when searching for a 
#   path to the polygon.
search_radius_km = 10

# Enter the desired forest stand characteristics as a SQL-style WHERE clause.
# You can copy this from the ArcGIS or QGIS "definition query".
VRI_where = """
    PROJ_AGE_1 >= 160 -- tree species 1 age is >= 160 years
    """

## Additional inputs
If you want to tweak the output a bit more, you can optionally edit these parameters: 
- Max number of candidates to display on the map
- Distance in meters between 

In [2]:
# Max Candidates: This many candidate forests will appear in the outputs
# You might want to increase this if the data is not very good in your region
max_candidates = 5

# Distance in meters between a VRI candidate centroid and the the nearest 
#   OpenStreetMap network node. You can think 
max_xcountry_distance_m = 500

# Optional: choose which fields to display on the final maps popups
# Dictionary where the keys are the field name and the values are the aliases for display on the final map
VRI_fields_dict = {
    "SPECIES_CD_1": "Dominant Tree Species Code(s)",
    "SPECIES_CD_2":  "Secondary Tree Species Code(s)",
    "PROJ_AGE_1": "Dominant Tree Species Age(s)"
}

# Analysis

In [3]:
# Install required package duckdb
%pip install duckdb

  Using cached duckdb-1.4.3-cp311-cp311-manylinux_2_26_x86_64.manylinux_2_28_x86_64.whl.metadata (4.3 kB)
Using cached duckdb-1.4.3-cp311-cp311-manylinux_2_26_x86_64.manylinux_2_28_x86_64.whl (20.5 MB)
Note: you may need to restart the kernel to use updated packages.


In [4]:
# Import functions and setup paths
import bcforestfinder as bcf
import pathlib
NOTEBOOK_PATH = pathlib.Path().resolve()
OUTPUT = NOTEBOOK_PATH / "outputs"

## Get start point geometry/buffer

In [5]:
# Get a EPSG:3005 geodataframe from geocoding the starting location string
start_point_gdf = bcf.get_starting_geom(starting_location_string)

# Generate the circle search area by buffering
start_point_circle = start_point_gdf.buffer(search_radius_km * 1000)

# Assert the geom is valid and bbox is legit
assert start_point_circle.geometry.is_valid.any(), "Try a different starting point"
# Check if the start point is within a bounding box containing all of BC
assert len(start_point_circle.cx[242359:1880910, 119152:1850376]) > 0, (
    "Your starting point appears to be outside BC."
    "Try entering WGS84 lat/long coordinates.")

## Selecting the VRI polygons

<div class="alert alert-block alert-info">
⚠️ <b>Common kernel error</b>
    
The following cell sometimes results in the kernel "dying unexpectedly". In this case, try restarting the notebook from the beginning and it will usually work on the second (or third) time. It usually takes about two minutes to run. 

One time it wouldn't work for me at all, but then when I tried again a few hours later it worked fine (without changing anything).

I'd like to try and solve this but I think it's a server-side issue.
</div>

In [6]:
# Return a geodataframe of the candidate VRI polygons
# Sometimes the kernel will die at this step - try restarting and run again. 
# Depending on your area this might take some time to run
vri_candidates = bcf.get_vri(VRI_where, VRI_fields_dict, start_point_circle)

# assert that the dataframe is not empty
assert len(vri_candidates) > 0, ("Warning: no candidate forest stands detected. "
    "Try one of the following methods:"
    "\nIncrease search radius"
    "\nCheck your attributes with the VRI data dictionary (linked above)")
# Merge adjacent polygons (dissolve)
vri_dissolve = bcf.dissolve_adjacent(vri_candidates)

## Find the fastest route from the start point to each candidate
Using openstreetmap data retrieved and analyzed with osmnx. The code is based on the example from the University of Helsinki [AutoGIS lesson](https://autogis-site.readthedocs.io/en/latest/lessons/lesson-6/network-analysis.html)

In [13]:
# Retrieve graph from openstreetmap using osmnx
graph = bcf.get_osm_network(search_radius_km, start_point_gdf)

print(f"Found {len(vri_dissolve)} candidate polygons before discarding candidates too far from nodes.")
# Get shortest route to each polygon
gdf_ranked = bcf.rank_gdf(graph, vri_dissolve, start_point_gdf, max_xcountry_distance_m)

# Join back to vri_dissolve
vri_dissolve_join = vri_dissolve.merge(gdf_ranked, left_index=True, right_on="polygon_id")

# Report top candidates
top_candidates = vri_dissolve_join.sort_values('rank').iloc[0:max_candidates].dropna()

assert len(top_candidates) > 0, (
    f"No routeable polygons were found from the {len(vri_dissolve)} candidates"
    " - try increasing the maximum distance from node.")

Found 12 candidate polygons before discarding candidates too far from nodes.
Discarded candidate 2.055km from node
Discarded candidate 1.309km from node
Discarded candidate 0.934km from node
Discarded candidate 0.793km from node
Discarded candidate 0.599km from node
Discarded candidate 0.982km from node
Discarded candidate 1.799km from node
Discarded candidate 0.681km from node


# Outputs
- List the candidates and their travel times
- Create and save Folium map

In [14]:
# List candidates and travel time as string:
for idx, i in top_candidates.iterrows():
    print(f"""{int(i['rank'])}: A {i.geometry.area/10000:.2f} ha stand with age(s) {i['PROJ_AGE_1']}.
    {i['travel_time_seconds']/60:.1f} minutes estimated travel time.
    {i['distance']/1000:.2f}km distance, {i['track_dis']/1000:.2f}km by rough track.
    Polygon centroid is {int(i['Cross-Country Distance'])} m from nearest network node.
    """)

1: A 35.89 ha stand with age(s) 177,172,182.
    10.5 minutes estimated travel time.
    10.13km distance, 0.00km by rough track.
    Polygon centroid is 387 m from nearest network node.
    
2: A 1.26 ha stand with age(s) 167.
    12.2 minutes estimated travel time.
    12.65km distance, 0.00km by rough track.
    Polygon centroid is 99 m from nearest network node.
    
3: A 6.50 ha stand with age(s) 167.
    12.6 minutes estimated travel time.
    13.05km distance, 0.00km by rough track.
    Polygon centroid is 169 m from nearest network node.
    
4: A 5.25 ha stand with age(s) 167.
    14.4 minutes estimated travel time.
    11.17km distance, 0.00km by rough track.
    Polygon centroid is 194 m from nearest network node.
    


## Folium Map

In [15]:
m = bcf.getFoliumMap(top_candidates, VRI_fields_dict, graph, start_point_gdf)

# Add title
import folium
map_title = "Nearest 160+ year old fores near Prince George, BC, ranked by travel time."
title_html = f'''
             <h3 align="center" style="font-size:20px"><b>{map_title}</b></h3>
             '''
m.get_root().html.add_child(folium.Element(title_html))

# Save the map
m.save(OUTPUT / 'PG_OldForest.html')

In [16]:
m